In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
from scipy.stats import mode



r_seed = 42

WORKDIR = 'C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

pd.set_option('display.float_format', lambda x: f'{x:.17f}')

C:\Users\cristiano.nascimento\AppData\Local\Temp\ipykernel_26424\4229836749.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\cristiano.nascimento\git\LSTM_giroscopio_acelerometro\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
WORKDIR = WORKDIR+'/Data/UCI-HAPT/RawData/'

In [4]:
print(WORKDIR)

C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/


In [5]:
import os
import pandas as pd
import numpy as np

def process_hapt_data(base_path='.'):
    """
    Process accelerometer and gyroscope data files
    
    Parameters:
    base_path (str): Directory containing the data files
    
    Returns:
    pandas.DataFrame: Processed HAPT data
    """
    # Initialize an empty list to store rows
    all_data_rows = []
    
    # Iterate through user and experiment numbers
    for user in range(1, 62):  # 01 to 61
        for exp in range(1, 124):  # Assuming 123 experiments
            # Pad numbers with zero
            user_str = f'{user:02d}'
            exp_str = f'{exp:02d}'
            
            # Construct file paths
            acc_file = os.path.join(base_path, f'acc_exp{exp_str}_user{user_str}.txt')
            gyro_file = os.path.join(base_path, f'gyro_exp{exp_str}_user{user_str}.txt')
            print(gyro_file)
            
            # Skip if files don't exist
            if not (os.path.exists(acc_file) and os.path.exists(gyro_file)):
                continue
            
            # Read files
            with open(acc_file, 'r') as acc, open(gyro_file, 'r') as gyro:
                acc_lines = acc.readlines()
                gyro_lines = gyro.readlines()
                
                # Verify same number of lines
                if len(acc_lines) != len(gyro_lines):
                    print(f"Mismatch in lines for user {user_str}, exp {exp_str}")
                    continue
                
                # Process each line
                for acc_line, gyro_line in zip(acc_lines, gyro_lines):
                    # Split and convert to float
                    acc_vals = [float(f'{float(val):.17f}') for val in acc_line.strip().split()]
                    gyro_vals = [float(f'{float(val):.17f}') for val in gyro_line.strip().split()]
                    
                    # Combine values
                    row_data = {
                        'acc_x': acc_vals[0],
                        'acc_y': acc_vals[1],
                        'acc_z': acc_vals[2],
                        'gyro_x': gyro_vals[0],
                        'gyro_y': gyro_vals[1],
                        'gyro_z': gyro_vals[2],
                        'user': user,
                        'experiment': exp
                    }
                    
                    all_data_rows.append(row_data)
    
    # Create DataFrame
    hapt_data_transformed = pd.DataFrame(all_data_rows)
    
    # Save to CSV
    hapt_data_transformed.to_csv('hapt_data_transformed.csv', index=False)
    
    return hapt_data_transformed

# Example usage
if __name__ == '__main__':
    result_df = process_hapt_data(WORKDIR)
    print(result_df.head())
    print(f"Total rows processed: {len(result_df)}")

C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp01_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp02_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp03_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp04_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp05_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp06_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp07_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp08_user01.txt
C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/Data/UCI-HAPT/RawData/gyro_exp09_user01.txt
C:/Users/cristiano.nascimento/git/LST

In [6]:
result_df.head()

,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,user,experiment
0,0.91805558987665181,-0.11249999942429351,0.50972225142938521,-0.05497787147760391,-0.06963863968849182,-0.03084869496524334,1,1
1,0.91111113046038117,-0.09305556168259389,0.53750004047060962,-0.01252273749560118,0.01924225501716137,-0.03848451003432274,1,1
2,0.88194449815976084,-0.08611111442228780,0.51388892707914757,-0.02351831272244453,0.27641651034355158,0.00641408516094089,1,1
3,0.88194449815976084,-0.08611111442228780,0.51388892707914757,-0.09346238523721695,0.36774086952209473,0.00122173049021512,1,1
4,0.87916671439325256,-0.10000000286491770,0.50555557577962285,-0.12431107461452480,0.47678032517433172,-0.02290744706988335,1,1


In [7]:
columns_to_convert = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

# Convert the selected columns to float
result_df[columns_to_convert] = result_df[columns_to_convert].astype(float)

In [14]:
result_df.head(300).tail()

,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,user,experiment
295,1.01805561097551589,-0.12083333856785380,0.10833333593049570,0.00458148913457990,-0.00977384392172098,0.01588249579071999,1,1
296,1.01527782720900794,-0.12500000206165160,0.10416667243669780,0.00427605677396059,-0.00885754637420177,0.00641408516094089,1,1
297,1.01666671909226203,-0.12083333856785380,0.10694444404724160,-0.00183259579353034,-0.00916297826915979,0.00916297826915979,1,1
298,1.02083339474202495,-0.12083333856785380,0.10416667243669780,-0.00458148913457990,-0.00946841109544039,0.01374446786940098,1,1
299,1.02222228662527903,-0.11805555480134550,0.09444444748786576,-0.00305432616733015,-0.00885754637420177,0.00946841109544039,1,1


In [10]:
result_df.shape

(1122772, 8)

In [9]:
print(result_df['acc_x'].iloc[0]) 

0.9180555898766518


In [15]:
import pandas as pd

# Load the labels.txt file into a DataFrame
labels = pd.read_csv(WORKDIR+'labels.txt', sep=' ', header=None, 
                     names=['experiment', 'user', 'activity', 'start', 'end'])

# Create a 'label' column in result_df with default NaN values
result_df['label'] = pd.NA

# Process each row in the labels DataFrame
for _, row in labels.iterrows():
    # Extract the interval and corresponding user and experiment
    start, end = row['start'], row['end']
    user, experiment = row['user'], row['experiment']
    activity = row['activity']
    
    # Find the index interval in result_df
    interval_mask = (result_df.index >= start) & (result_df.index <= end)
    
    # Verify if user and experiment match
    if all(result_df.loc[interval_mask, 'user'] == user) and \
       all(result_df.loc[interval_mask, 'experiment'] == experiment):
        # Assign the activity label to the 'label' column in the specified interval
        result_df.loc[interval_mask, 'label'] = activity


In [25]:
result_df.head(1235).tail()

,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,user,experiment,label
1230,1.03055563792480398,-0.14444445195931571,0.07916666716198170,-0.03481931984424591,0.07666359096765518,0.02260201424360275,1,1,5
1231,1.01666671909226203,-0.14305556007606160,0.05833333753702795,-0.00610865233466029,0.03298672288656235,0.00977384392172098,1,1,5
1232,1.01527782720900794,-0.14166666819280749,0.05972222334229981,0.03145955875515938,0.00335975876078010,0.01496619824320078,1,1,5
1233,1.02083339474202495,-0.16388888970101539,0.05555555984850197,0.07299839705228806,0.03390302136540413,-0.00305432616733015,1,1,7
1234,1.02777785415829492,-0.17361112072782969,0.06944444829113189,0.10384709388017650,0.06414084881544113,0.00519235478714108,1,1,7


In [ ]:
labels_train = pd.read_csv(WORKDIR+'/Data/UCI-HAPT/Train/y_train.txt',sep=' ',header=None)


In [ ]:
labels_train.head()

In [ ]:
labels_train.columns=['atividade']

In [ ]:
labels_train["atividade"].value_counts().sort_index()

In [ ]:
labels_train.shape

In [ ]:
X_validation_data = pd.read_csv(WORKDIR+'/Data/UCI-HAPT/Test/X_test.txt',sep=' ',header=None, dtype={"value": "float64"})

In [ ]:
X_validation_data.columns = final_columns

In [ ]:
X_validation_data.head()

In [ ]:
X_validation_data.shape

In [ ]:
labels_val = pd.read_csv(WORKDIR+'/Data/UCI-HAPT/Test/y_test.txt',sep=' ',header=None)


In [ ]:
labels_val.head()

In [ ]:
labels_val.columns=['atividade']

In [ ]:
labels_val.shape

In [ ]:
X_validation_data.head()

In [ ]:
X_validation_data.shape

In [ ]:
X_train_data.isnull().sum()

In [ ]:
X_train_data.shape

In [ ]:
#remove the null class
X_train_data=X_train_data.dropna()

In [ ]:
X_train_data.shape

In [ ]:
labels_val["atividade"].value_counts().sort_index()

# **Data Exploration**

In [ ]:
X_train_data = X_train_data.reset_index(drop=True)

In [ ]:
labels_train = labels_train.reset_index(drop=True)

In [ ]:
#selecting only activities 0 to 3
labels_train = labels_train[labels_train["atividade"].isin([4,5,6,7,8,9,10])]

In [ ]:
# Define the mapping dictionary
mapping_dict = {4:0,5:1,6:2,7:3,8:4,9:5, 10:6}

In [ ]:
labels_train["atividade"] = labels_train["atividade"].replace(to_replace=mapping_dict)

In [ ]:
filtered_indices_labels_train = labels_train.index

In [ ]:
X_train_data = X_train_data.iloc[filtered_indices_labels_train]

In [ ]:
X_train_data.shape

In [ ]:
labels_train.shape

In [ ]:
X_validation_data = X_validation_data.reset_index(drop=True)

In [ ]:
X_validation_data.shape

In [ ]:
labels_val.shape

In [ ]:
#selecting only activities 0 to 3
labels_val = labels_val[labels_val["atividade"].isin([4,5,6,7,8,9,10])]

In [ ]:
# Define the mapping dictionary
mapping_dict = {4:0,5:1,6:2,7:3,8:4,9:5, 10:6}

In [ ]:
labels_val.shape

In [ ]:
labels_val["atividade"] = labels_val["atividade"].replace(to_replace=mapping_dict)

In [ ]:
labels_val.shape

In [ ]:
filtering_indices_labels_val = labels_val.index

In [ ]:
filtering_indices_labels_val.shape

In [ ]:
X_validation_data = X_validation_data.iloc[filtering_indices_labels_val]

In [ ]:
X_validation_data.shape


In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

def balance_data_with_smote(_x_data, label_data):
    """
    Balances the class distribution in a DataFrame using SMOTE.

    Parameters:
        data (pd.DataFrame): The input DataFrame containing features and labels.
        label_column (str): The name of the column containing class labels.

    Returns:
        pd.DataFrame: A balanced DataFrame with the same structure as the input.
    """
    # Separate features and labels
    X = _x_data
    y = label_data

    # Check class distribution before SMOTE
    print("Before SMOTE:", Counter(y))

    # Apply SMOTE
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    # Check class distribution after SMOTE
    print("After SMOTE:", Counter(y_smote)) 
    print(X_smote.shape) 
    print(X_smote.shape) 

    return X_smote, y_smote

# Example usage:
# balanced_df = balance_data_with_smote(data, label_column='label')


In [ ]:
X_train_data,labels_train = balance_data_with_smote(X_train_data,labels_train)

In [ ]:
labels_train.shape

In [ ]:
X_train_data.shape

# **Models**

In [ ]:
labels_val_categorical = to_categorical(labels_val,num_classes=7)

In [ ]:
print(labels_val_categorical)

In [ ]:
print(labels_val_categorical.shape)

In [ ]:
#print(occurrence_transitions_slide)

In [ ]:
labels_train_categorical = to_categorical(labels_train,num_classes=7)

In [ ]:
print(labels_train_categorical)

In [ ]:
print(labels_train_categorical.shape)

In [ ]:
X_train = X_train_data
y_train = labels_train_categorical
X_validation = X_validation_data
y_validation = labels_val_categorical

In [ ]:
X_train = X_train.values

In [ ]:
X_train = np.expand_dims(X_train, axis=-2) 

In [ ]:
X_train.shape

In [ ]:
X_validation = X_validation.values

In [ ]:
X_validation = np.expand_dims(X_validation, axis=-2) 

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [ ]:
#X_train,y_train = shuffle_data(X_train,y_train)
#X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
#from tensorflow.keras import regularizers

In [ ]:
''' def cria_LSTM(hp):
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,6), return_sequences=False))
  model.add(tf.keras.layers.Dense(hp.Int("input_units_relu",min_value=32,max_value=256,step=32), activation='relu'))
  model.add(tf.keras.layers.BatchNormalization())  
  model.add(tf.keras.layers.Dense(units=4, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
  return model
 '''

In [ ]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor(hp):
    inputs = Input(shape=(1,561))
    
    # LSTM layer
    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), 
                    kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    return_sequences=True)(inputs)
    
    
    dense_out = Dense(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), activation='relu')(lstm_out)
    # Attention mechanism using Keras Attention layer
    attention = Attention()([dense_out, dense_out])
    
    # Context vector computation
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(7, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 
                                                              min_value=1e-4, 
                                                              max_value=1e-2, 
                                                              sampling='LOG', 
                                                              default=1e-3)),
                  metrics=['accuracy'])
    
    return model


In [ ]:
""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape
import keras.backend as K

def lstm_attention_sensor(hp):
    inputs = Input(shape=(200,6))
    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
             recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),return_sequences=True)(inputs)
    
    # Attention mechanism
    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation='relu')(lstm_out)    
    attention = Permute([2, 1])(attention)
    attention = Activation('softmax')(attention)
    attention = Permute([2, 1])(attention)
    
    context_vector = Multiply()([lstm_out, attention])
    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    
    output = Dense(6, activation='softmax')(context_vector)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
    return model
 """

In [ ]:
tuner = RandomSearch(
    lstm_attention_sensor,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 5,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [ ]:
tuner.search(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=32, use_multiprocessing=True)

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.history

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
print(segments_test)

In [ ]:
print(labels_test_categorical.shape)

In [ ]:
print(labels_test_categorical)

In [ ]:
loss_test, accuracy_test = best_model.evaluate(segments_test, labels_test_categorical)

In [ ]:
predictions_test = best_model.predict(segments_test)

In [ ]:
print(predictions_test)

In [ ]:
classes_x_test=np.argmax(predictions_test,axis=1)

In [ ]:
print(classes_x_test)

In [ ]:
labels_test_argmax = np.argmax(labels_test_categorical,axis=1)

In [ ]:
print(labels_test_argmax)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm_test = confusion_matrix(labels_test_argmax, classes_x_test)

In [ ]:
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot()

In [ ]:
print(classification_report(labels_test_argmax, classes_x_test))

In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved_"+data_hora)